In [4]:
import torch
from RNN import RNN
from utils_RNN import params
import pickle
import io
from traffic_data_loader import data_loader_full, tensor_reshape, Traffic_Flow_Data
import numpy as np

In [5]:
# retreive parameters from params
input_size = params['input_size']
hidden_size = params['hidden_size']
num_layers = params['num_layers']
output_size = params['output_size']
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [6]:
# load model
model_LSTM = RNN(input_size, hidden_size, num_layers, output_size).to(device)
model_LSTM.load_state_dict(torch.load('saved_model/model_RNN.pth', map_location=device))
model_LSTM.eval() # enable evaluation mode

RNN(
  (rnn): RNN(17, 512, num_layers=3, batch_first=True)
  (fc): Linear(in_features=512, out_features=17, bias=True)
)

In [7]:
# start prediction
class CPU_Unpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if module == 'torch.storage' and name == '_load_from_bytes':
            return lambda b: torch.load(io.BytesIO(b), map_location='cpu')
        else:
            return super().find_class(module, name)


file_path = '../Model_Final/Predicted/results.pkl'
with open(file_path, 'rb') as pickle_file:
    pred = CPU_Unpickler(pickle_file).load()
data = pred['flow_recon'].to(device)

data_occupancy_all, data_flow_all, data_speed_all = data_loader_full()
X_occu_all, _ = data_occupancy_all[:, :2], data_occupancy_all[:, 2]
X_occu_all = torch.tensor(X_occu_all, dtype=torch.float32).to(device)

data = torch.cat((X_occu_all, data), dim=1).detach()

data = tensor_reshape(data)

data_validation = data[int(data.size(0) * 0.6):, :]
# data_validation = data

data_val = Traffic_Flow_Data(data_validation, window_size = params['window_size'])

In [8]:
# define a function to predict n steps
def predict_n_steps(model, data_val, n_steps):
    """
    model: trained LSTM model
    data_val: Traffic_Flow_Data object (validation dataset)
    n_steps: number of recursive prediction steps
    """
    model.eval()
    device = next(model.parameters()).device

    Y_VAL = []
    Y_PRED = []

    with torch.no_grad():
        for idx in range(len(data_val) - n_steps):
            x_val, _ = data_val[idx]
            x_val = x_val.unsqueeze(0).to(device)

            # Recursive prediction
            for step in range(n_steps):
                y_pred = model(x_val)
                y_pred_expand = y_pred.unsqueeze(0)  # (1, 1, feature_dim)

                # Update input for next step
                x_val = torch.cat((x_val, y_pred_expand), dim=1)
                x_val = x_val[:, 1:, :]  # remove the oldest time step

            # After n_steps prediction, collect y_pred and corresponding ground truth
            y_pred = y_pred.cpu().detach().numpy()
            y_pred = np.squeeze(y_pred, axis=0)

            _, y_val = data_val[idx + n_steps]
            y_val = y_val.cpu().detach().numpy()

            Y_VAL.append(y_val)
            Y_PRED.append(y_pred)

    Y_VAL = np.vstack(Y_VAL)
    Y_PRED = np.vstack(Y_PRED)

    return Y_VAL, Y_PRED

In [9]:
n_steps = 1
Y_VAL_1, Y_PRED_1 = predict_n_steps(model_LSTM, data_val, n_steps)

rmse_1 = np.sqrt(np.nanmean((Y_VAL_1 - Y_PRED_1) ** 2))
mape_1 = np.nanmean(np.abs((Y_VAL_1 - Y_PRED_1) / Y_VAL_1)) * 100

print(f'RMSE ({n_steps}-step): {rmse_1:.4f}')
print(f'MAPE ({n_steps}-step): {mape_1:.2f}%')

RMSE (1-step): 7.3138
MAPE (1-step): 10.02%


In [10]:
n_steps = 2
Y_VAL_2, Y_PRED_2 = predict_n_steps(model_LSTM, data_val, n_steps)

rmse_2 = np.sqrt(np.nanmean((Y_VAL_2 - Y_PRED_2) ** 2))
mape_2 = np.nanmean(np.abs((Y_VAL_2 - Y_PRED_2) / Y_VAL_2)) * 100

print(f'RMSE ({n_steps}-step): {rmse_2:.4f}')
print(f'MAPE ({n_steps}-step): {mape_2:.2f}%')

RMSE (2-step): 7.7941
MAPE (2-step): 10.76%


In [11]:
n_steps = 3
Y_VAL_3, Y_PRED_3 = predict_n_steps(model_LSTM, data_val, n_steps)

rmse_3 = np.sqrt(np.nanmean((Y_VAL_3 - Y_PRED_3) ** 2))
mape_3 = np.nanmean(np.abs((Y_VAL_3 - Y_PRED_3) / Y_VAL_3)) * 100

print(f'RMSE ({n_steps}-step): {rmse_3:.4f}')
print(f'MAPE ({n_steps}-step): {mape_3:.2f}%')

RMSE (3-step): 8.6883
MAPE (3-step): 12.02%


In [12]:
n_steps = 4
Y_VAL_4, Y_PRED_4 = predict_n_steps(model_LSTM, data_val, n_steps)

rmse_4 = np.sqrt(np.nanmean((Y_VAL_4 - Y_PRED_4) ** 2))
mape_4 = np.nanmean(np.abs((Y_VAL_4 - Y_PRED_4) / Y_VAL_4)) * 100

print(f'RMSE ({n_steps}-step): {rmse_4:.4f}')
print(f'MAPE ({n_steps}-step): {mape_4:.2f}%')

RMSE (4-step): 9.9615
MAPE (4-step): 13.75%


In [13]:
n_steps = 5
Y_VAL_5, Y_PRED_5 = predict_n_steps(model_LSTM, data_val, n_steps)

rmse_5 = np.sqrt(np.nanmean((Y_VAL_5 - Y_PRED_5) ** 2))
mape_5 = np.nanmean(np.abs((Y_VAL_5 - Y_PRED_5) / Y_VAL_5)) * 100

print(f'RMSE ({n_steps}-step): {rmse_5:.4f}')
print(f'MAPE ({n_steps}-step): {mape_5:.2f}%')

RMSE (5-step): 11.1766
MAPE (5-step): 15.35%


In [14]:
import pandas as pd

In [15]:
data = {
    'Prediction Length': ['RMSE(%)', 'MAPE(%)'],
    '3-min prediction': [rmse_1, mape_1],
    '6-min prediction': [rmse_2, mape_2],
    '9-min prediction': [rmse_3, mape_3],
    '12-min prediction': [rmse_4, mape_4],
    '15-min prediction': [rmse_5, mape_5]
}
df = pd.DataFrame(data)
df.iloc[:, 1:] = df.iloc[:, 1:].round(2)
df.to_csv('Tables/Prediction_Error_RNN.csv', index=False)